# Key Repeat JS

> Custom key repeat engine with configurable initial delay, repeat interval, and throttle floor.

In [ ]:
#| default_exp js.repeat

In [ ]:
#| export
from cjm_fasthtml_token_selector.core.config import TokenSelectorConfig

In [ ]:
#| export
def _generate_movement_dispatch_js(
    config:TokenSelectorConfig,  # config for this instance
) -> str:  # JS code fragment for the movement dispatcher
    """Generate the key-to-movement dispatch logic."""
    left_key = config.left_key
    right_key = config.right_key
    mode = config.selection_mode

    # Base movement for unmodified keys
    base_dispatch = f"""
        if (key === '{left_key}') return ns.moveLeft;
        if (key === '{right_key}') return ns.moveRight;
"""

    # Shift-modified keys for span extend
    if mode == "span":
        shift_dispatch = f"""
        if (key === '{left_key}') return ns.extendLeft;
        if (key === '{right_key}') return ns.extendRight;
"""
    else:
        shift_dispatch = """
        return null;
"""

    return f"""
    ns._getMovementFn = function(key, shiftKey) {{
        if (shiftKey) {{{shift_dispatch}
        }}
        {base_dispatch}
        return null;
    }};
"""

In [ ]:
#| export
def generate_key_repeat_js(
    config:TokenSelectorConfig,  # config with timing settings
) -> str:  # JS code fragment for the IIFE
    """Generate the custom key repeat engine JS."""
    initial_delay = config.initial_delay
    repeat_interval = config.repeat_interval
    throttle_floor = config.throttle_floor

    dispatch = _generate_movement_dispatch_js(config)

    return dispatch + f"""
    // Key repeat engine state
    ns._repeatTimer = null;
    ns._repeatKey = null;
    ns._repeatShift = false;
    ns._lastMoveTime = 0;

    ns._handleKeyDown = function(e) {{
        if (!ns.active) return;

        var moveFn = ns._getMovementFn(e.key, e.shiftKey);
        if (!moveFn) return;

        e.preventDefault();
        e.stopPropagation();

        // If same key already held, let the timer run
        if (ns._repeatKey === e.key && ns._repeatShift === e.shiftKey) return;

        // Cancel any existing repeat
        if (ns._repeatTimer) {{
            clearTimeout(ns._repeatTimer);
            ns._repeatTimer = null;
        }}

        ns._repeatKey = e.key;
        ns._repeatShift = e.shiftKey;

        // Immediate first movement
        var now = Date.now();
        if (now - ns._lastMoveTime >= {throttle_floor}) {{
            moveFn();
            ns._lastMoveTime = now;
        }}

        // Start repeat after initial delay, then switch to interval
        ns._repeatTimer = setTimeout(function repeatTick() {{
            var fn = ns._getMovementFn(ns._repeatKey, ns._repeatShift);
            if (!fn || !ns.active) {{
                ns._repeatKey = null;
                ns._repeatShift = false;
                return;
            }}
            var tickNow = Date.now();
            if (tickNow - ns._lastMoveTime >= {throttle_floor}) {{
                fn();
                ns._lastMoveTime = tickNow;
            }}
            ns._repeatTimer = setTimeout(repeatTick, {repeat_interval});
        }}, {initial_delay});
    }};

    ns._handleKeyUp = function(e) {{
        if (e.key === ns._repeatKey) {{
            if (ns._repeatTimer) {{
                clearTimeout(ns._repeatTimer);
                ns._repeatTimer = null;
            }}
            ns._repeatKey = null;
            ns._repeatShift = false;
        }}
    }};
"""

In [ ]:
from cjm_fasthtml_token_selector.core.config import TokenSelectorConfig, _reset_prefix_counter

_reset_prefix_counter()

# Default timing
cfg = TokenSelectorConfig(prefix="t")
js = generate_key_repeat_js(cfg)
assert "ns._handleKeyDown" in js
assert "ns._handleKeyUp" in js
assert "ns._repeatTimer" in js
assert str(cfg.initial_delay) in js
assert str(cfg.repeat_interval) in js
assert str(cfg.throttle_floor) in js
assert "ArrowLeft" in js
assert "ArrowRight" in js
print("Default key repeat JS generated!")

# Custom keys
cfg_wasd = TokenSelectorConfig(prefix="tw", left_key="a", right_key="d")
js_wasd = generate_key_repeat_js(cfg_wasd)
assert "'a'" in js_wasd
assert "'d'" in js_wasd
print("WASD key repeat JS generated!")

# Span mode includes shift dispatch
cfg_span = TokenSelectorConfig(prefix="ts", selection_mode="span")
js_span = generate_key_repeat_js(cfg_span)
assert "ns.extendLeft" in js_span
assert "ns.extendRight" in js_span
print("Span mode key repeat JS generated!")

# Custom timing
cfg_fast = TokenSelectorConfig(prefix="tf", initial_delay=100, repeat_interval=30, throttle_floor=20)
js_fast = generate_key_repeat_js(cfg_fast)
assert "100" in js_fast
assert "30" in js_fast
assert "20" in js_fast
print("Custom timing key repeat JS generated!")

_reset_prefix_counter()
print("All key repeat JS tests passed!")

Default key repeat JS generated!
WASD key repeat JS generated!
Span mode key repeat JS generated!
Custom timing key repeat JS generated!
All key repeat JS tests passed!


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()